In [2]:
import os
os.chdir('../')

In [3]:
import numpy as np
from utils.fid import FIDInception


data = np.load('/dataset/dit/VIRTUAL_imagenet256_labeled.npz')
data['arr_0'].shape

(10000, 256, 256, 3)

In [5]:
import numpy as np, torch
from PIL import Image
from tqdm.auto import tqdm
from utils.fid import FIDInception  # 위에서 정의한 클래스 그대로 사용

NPZ  = '/dataset/dit/VIRTUAL_imagenet256_labeled.npz'  # arr_0 포함
OUT  = '/dataset/dit/valid_stats.pt'
B    = 1024  # 배치 크기

# 1) 레퍼런스 로드
arr = np.load(NPZ)['arr_0']  # (N,256,256,3) uint8 [0..255]
N = len(arr)

# 2) Inception 준비 (forward 사용 → normalize_input=True)
fid = FIDInception(dims=2048, normalize_input=True).eval()
for p in fid.parameters(): p.requires_grad_(False)

# 3) 특징 추출 (tqdm 진행바)
feats = []
with torch.no_grad():
    for i in tqdm(range(0, N, B), total=(N + B - 1)//B, desc="Extracting Inception features"):
        j = min(i + B, N)
        pil_batch = [Image.fromarray(arr[k], 'RGB') for k in range(i, j)]
        f = fid(pil_batch).cpu().numpy()  # forward 사용
        feats.append(f)
feats = np.concatenate(feats, axis=0).astype(np.float64)  # [N,2048]

# 4) μ/Σ 계산
mu    = feats.mean(0)
sigma = np.cov(feats, rowvar=False)

# 5) .pt로 저장 (torch.save)
torch.save({'mu': torch.from_numpy(mu), 'sigma': torch.from_numpy(sigma)}, OUT)
OUT


Extracting Inception features:   0%|          | 0/10 [00:00<?, ?it/s]/tmp/ipykernel_68719/883347689.py:23: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  pil_batch = [Image.fromarray(arr[k], 'RGB') for k in range(i, j)]
Extracting Inception features: 100%|██████████| 10/10 [00:09<00:00,  1.02it/s]


'/dataset/dit/valid_stats.pt'